# Experiment 10: Retrieval Augmented Chatbot over the IPC

CSL75 Skill Enhancement Laboratory.

A chatbot that answers questions about a statute by retrieving the governing
provisions and answering only from them, citing section numbers.

| Step | Requirement |
| --- | --- |
| i | Load and preprocess the document, PDF text extraction |
| ii | Split into chunks for efficient retrieval |
| iii | Embedding based vector store for semantic search |
| iv | Retrieval based QA pipeline with a language model |
| v | Chatbot interface |

Set the runtime to a T4 GPU before running.

## 1. Install dependencies

The version pins matter. transformers 5.x removed pipeline tasks this project
uses, and gradio 6.x pulls a huggingface-hub that breaks transformers 4.x.

In [ ]:
!pip install -q "transformers>=4.44,<5" "sentence-transformers>=3,<6" \
    "huggingface-hub>=0.34,<1.0" "langchain-community>=0.3" "langchain-huggingface>=0.1" \
    "faiss-cpu>=1.8" "pypdf>=4.0" "gradio>=4.44,<6" "cohere>=5.11" "accelerate>=0.33"

## 2. Get the code

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/rhuzaii/genaitut.git"
ROOT = Path("/content/genaitut")
PROJECT = ROOT / "experiment10_ipc_chatbot"
MODULES = ("config", "ingest", "chunking", "vectorstore", "chatbot",
           "evaluate", "build_index", "ask", "app")

if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--quiet", REPO, str(ROOT)], check=True)

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
for name in MODULES:
    if name in sys.modules:
        importlib.reload(sys.modules[name])

print("working in", PROJECT)
print("modules  :", sorted(f.name for f in PROJECT.glob("*.py")))

## 3. Supply the IPC

Download the Indian Penal Code from https://www.indiacode.nic.in and upload it
here. Skip this cell to use `documents/sample_statute.txt`, a fictional test
statute that is **not law** and exists only so the pipeline can be verified.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if uploaded:
    name = next(iter(uploaded))
    Path("documents").mkdir(exist_ok=True)
    Path("documents/ipc.pdf").write_bytes(uploaded[name])
    print("saved documents/ipc.pdf from", name)
else:
    print("nothing uploaded, the fictional sample statute will be used")

## 4. Steps i to iii: load, chunk, index

In [ ]:
import build_index

build_index.main(["--index-dir", "index", "--strategy", "section"])

## 5. Language model credentials

A Cohere trial key from dashboard.cohere.com gives the best answers. Press
Enter to skip and use a local model instead, or set the backend to
`extractive` later to run with no language model at all.

In [ ]:
import getpass
import os

key = getpass.getpass("Cohere API key (press Enter to skip): ").strip()
if key:
    os.environ["COHERE_API_KEY"] = key
    print("Cohere backend will be used.")
else:
    os.environ.pop("COHERE_API_KEY", None)
    print("No key set. A local model will be used.")

## 6. Step iv: ask questions

Note what happens on the last question. Nothing in the document addresses it,
so the bot declines instead of answering from general knowledge.

In [ ]:
import chatbot
from config import Config

cfg = Config(index_dir="index")
bot = chatbot.Chatbot(cfg)
print("backend:", bot.backend)

QUESTIONS = [
    "What is the punishment for ragging?",
    "How long do I have to appeal a decision?",
    "What is the capital of France?",
]

for question in QUESTIONS:
    answer = bot.ask(question)
    print("=" * 84)
    print("Q:", question)
    print(answer.text)
    if answer.sources and not answer.refused:
        print()
        print(answer.formatted_sources())
    print()

### Conversational memory

The follow up has no subject of its own. It is rewritten against the history
before retrieval, which is what makes it answerable.

In [ ]:
history = []
for question in ["What is the punishment for ragging?", "Does consent matter?"]:
    answer = bot.ask(question, history)
    print("Q:", question)
    if answer.standalone_question != question:
        print("   rewritten as:", answer.standalone_question)
    print(answer.text)
    print()
    history.append((question, answer.text))

## 7. Evaluation

Retrieval is measured separately from generation, because a section that is
never retrieved can never be cited no matter how good the language model is.
Most end to end failures in a RAG system start in retrieval.

Replace `eval_questions.json` with your own annotated questions once the IPC
is indexed. The shipped set describes the fictional sample statute.

In [ ]:
import ask

ask.main(["--index-dir", "index", "--backend", "extractive", "--evaluate"])

In [ ]:
ask.main(["--index-dir", "index", "--backend", "extractive", "--compare-chunking"])

## 8. Step v: the chatbot interface

`share=True` gives a public link, which is how you open it from Colab. The
link is temporary and expires when the cell stops.

In [ ]:
import app
from config import Config

cfg = Config(index_dir="index")
demo = app.build_interface(cfg)
demo.launch(share=True)

## 9. Collect the outputs for the report

In [ ]:
import json
import shutil
from pathlib import Path

from google.colab import files

manifest = json.loads((Path("index") / "index_manifest.json").read_text())
print(json.dumps(manifest, indent=2))

shutil.make_archive("experiment10_index", "zip", "index")
files.download("experiment10_index.zip")